In [121]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

CUDA available: True
GPU name: NVIDIA H100 80GB HBM3 MIG 1g.10gb


## 1. INSTALLS, IMPORTS, AND PARAMETERS

### --- 1. Installs ---

In [122]:
# all the required installs have been moved to requirements.txt
# chose venv for execution otherwise it's pointless ._.

### --- 2. Imports ---

In [123]:
# ---- Standard libraries ----
import os
import sys
import json
import csv
import time
import math
import ast
import copy
import heapq
import random
import pickle
import logging
from datetime import datetime
from collections import defaultdict, namedtuple, deque
from itertools import permutations

# ---- Data manipulation / visualization ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tqdm import tqdm

# ---- Geo-related ----
import geojson
import geopandas as gpd
from shapely.geometry import Point, Polygon
from alphashape import alphashape
import geopy.distance
import networkx as nx
import requests
import overpass  # Overpass API for OpenStreetMap queries

# ---- PyTorch (for RL/deep learning parts) ----
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ---- Removed: this line (was for Colab only) ----
# from google.colab import drive


### --- 3. Global Parameters ---

In [124]:
state_name = "Ohio"
city_name = "Columbus"
mini = True

### 1. Setup

In [125]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

# Local data folder
DATA_DIR = PROJECT_ROOT / "Delivery_Data"
DATA_DIR.mkdir(exist_ok=True)

print(f"Using data directory: {DATA_DIR}")

Using data directory: /root/UMST_approach/Delivery_Data


In [126]:
# --- 5. Set Directories (local/cloud version) ---
from pathlib import Path
import os

try:
    # Step 1: base path (one level up from "Source Code/")
    personal_dir = str(Path.cwd().parent / "Delivery_Data") + "/"

    # Step 2: data folder based on city_name and mini flag
    data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"

    # Step 3: output folder (same structure as before)
    output_dir = data_dir + "/UMST Graph/maddpg_baseline"

    # Step 4: create output directory if missing
    os.makedirs(output_dir, exist_ok=True)

    # Step 5: print info
    print(f"Data directory: {data_dir}")
    print(f"Output directory: {output_dir}")

    # Step 6: list contents for sanity check
    data = os.listdir(data_dir)
    print(f"Files in data_dir: {data}")

except Exception as e:
    print(f"Error setting up directories: {e}")
    print(f"Personal Dir Path: {personal_dir}")
    print(f"Data Dir Path: {data_dir}")


Data directory: /root/UMST_approach/Delivery_Data/Columbus_mini - RL Delivery Data
Output directory: /root/UMST_approach/Delivery_Data/Columbus_mini - RL Delivery Data/UMST Graph/maddpg_baseline
Files in data_dir: ['Images', 'Deliveries', 'Processed Location Data', 'Q Tables', 'avg_hotspot_data.json', 'Census Data', 'results_fixed_buffer', 'results_multiply', 'gh_cache', 'UMST Graph', 'Original Location Data', 'results_rangewise', 'results_randomized', 'Hotspot Data']


# 2. RE-USABLE DATA & ENVIRONMENT CLASSES (THE "WORLD")
 This is to keep the deliveries same as they were even when operating on two different methods. This follows the same structure as used in the file `V7_02_order_bundling.ipynb`

#### 1. Delivery Class


In [127]:
INTERVALS = [900, 1200, 1500, 1800, 2700, 3600, 5400, 7200, 10800, 14400]

In [128]:
class Delivery:
    """
    Represents a single delivery request.
    (This is the simplified version from your notebook)
    """
    id_counter = 0

    def __init__(self, start_node, end_node, start_time, time_tolerance_factor,
                 shortest_path=None):
        # Identity
        self.id = Delivery.id_counter
        Delivery.id_counter += 1

        # Route (immutable)
        self.start_node = start_node
        self.end_node = end_node
        self.shortest_path = shortest_path if shortest_path else [start_node, end_node]

        # Time constraints
        self.start_time = start_time
        self.TIME_TOLERANCE_FACTOR = time_tolerance_factor
        self.time_limit = None  # Set after initialization

        # Current state (mutable)
        self.current_node = start_node
        self.path_index = 0
        self.in_transition = False
        self.time_till_next_node = 0
        self.wait_time_remaining = 0 # Used for heuristic baseline, but good to keep

        # Completion tracking
        self.completed = False
        self.successful = False
        self.end_time = None

        # Statistics
        self.distance_traveled = 0
        self.actual_path = [start_node]
        self.num_vehicle_changes = 0 # We'll re-purpose this for the RL agent
        self.times_bundled = 0

        # Bundle reference (managed by Bundle class or RL Env)
        self.current_bundle_id = None

    def get_next_node(self):
        """Get next node in planned path."""
        if self.path_index < len(self.shortest_path) - 1:
            return self.shortest_path[self.path_index + 1]
        return None

    def move_to_next_node(self, distance, travel_time):
        """Start transition to next node."""
        self.in_transition = True
        self.time_till_next_node = travel_time
        self.distance_traveled += distance

    def arrive_at_node(self, node):
        """Complete arrival at node."""
        self.in_transition = False
        self.current_node = node
        self.actual_path.append(node)
        self.path_index += 1

        # CRITICAL: Check if reached destination
        if self.current_node == self.end_node:
            self.completed = True
            # Note: successful status set later

    def reset(self):
        """Reset to initial state."""
        self.current_node = self.start_node
        self.path_index = 0
        self.in_transition = False
        self.time_till_next_node = 0
        self.wait_time_remaining = 0
        self.completed = False
        self.successful = False
        self.end_time = None
        self.distance_traveled = 0
        self.actual_path = [self.start_node]
        self.num_vehicle_changes = 0
        self.current_bundle_id = None

    def __repr__(self):
        status = "✅" if self.completed else ("🚗" if self.in_transition else "⏳")
        bundle = f"[B{self.current_bundle_id}]" if self.current_bundle_id else ""
        return f"D{self.id}:{self.start_node}→{self.end_node}|{status}{bundle}@N{self.current_node}"

#### 2. Graph & Pathfinding Helpers

In [129]:
def build_adjacency_matrix(graph: nx.Graph, tract_to_index, dist_attr='distance', time_attr='time'):
    """
    Create adjacency dict for direct edges only using integer indices.
    Each node maps to a list of (neighbor_index, distance, time_seconds)
    """
    adjacency = {}

    for u, v, data in graph.edges(data=True):
        u_idx = tract_to_index[u]
        v_idx = tract_to_index[v]
        dist = float(data.get(dist_attr, 0))
        time = float(data.get(time_attr, 0)) * 60  # minutes -> seconds
        adjacency.setdefault(u_idx, []).append((v_idx, dist, time))
        adjacency.setdefault(v_idx, []).append((u_idx, dist, time))

    num_edges = sum(len(v) for v in adjacency.values()) // 2
    print(f"✓ Adjacency built: {len(adjacency)} nodes with {num_edges} edges")
    return adjacency


def astar_shortest_path(adjacency, start, goal, node_positions=None, weight_type='time'):
    """
    Compute shortest path from start → goal using A*.
    - adjacency: dict[index] -> list of (neighbor_index, dist, time)
    - node_positions: dict[index] -> (lat, lon) for heuristic
    - weight_type: 'time' or 'distance'
    Returns: (total_cost, path_indices)
    """
    def heuristic(u, v):
        if node_positions is None:
            return 0
        (x1, y1), (x2, y2) = node_positions[u], node_positions[v]
        return math.hypot(x1 - x2, y1 - y2)

    frontier = [(0, start)]
    came_from = {start: None}
    cost_so_far = {start: 0}

    while frontier:
        _, current = heapq.heappop(frontier)
        if current == goal:
            break

        if adjacency.get(current) is None:
            continue # Node has no outgoing edges

        for neighbor, dist, time in adjacency.get(current, []):
            weight = time if weight_type == 'time' else dist
            new_cost = cost_so_far[current] + weight

            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                priority = new_cost + heuristic(neighbor, goal)
                heapq.heappush(frontier, (priority, neighbor))
                came_from[neighbor] = current

    # Reconstruct path
    if goal not in came_from:
        return float('inf'), []

    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()

    return cost_so_far[goal], path

#### 3. DeliveryList Class (The Task Generator)

(This is UNCHANGED from the original file)
This is CRUCIAL for ensuring we test both models on the *exact same* deliveries.

In [130]:
class DeliveryList:
    """Manages collection of deliveries with shortest path calculation."""

    def __init__(self, max_delivery_time, load, time_tolerance_factor,
                 hours=1, peaks=[0.25, 0.75], sigma=10, num_hotspots=50,
                 adjacency_matrix=None, node_positions=None):

        self.load = load
        self.hours = hours
        self.peaks = peaks
        self.sigma = sigma
        self.time_tolerance_factor = time_tolerance_factor
        self.max_delivery_time = max_delivery_time
        self.num_hotspots = num_hotspots
        self.adjacency_matrix = adjacency_matrix
        self.node_positions = node_positions

        if adjacency_matrix is None or node_positions is None:
            raise ValueError("adjacency_matrix and node_positions are required!")

        print("📊 Generating temporal distribution...")
        self.distribution, self.loads = self.generate_distribution()

        print("🗺️  Pre-computing all shortest paths...")
        self.precomputed_paths = {}  # (start, end) -> (travel_time, path)
        self.valid_pairs = []  # List of (start, end) pairs within time limit
        self.precompute_all_paths()

        print("📦 Generating deliveries with pre-computed paths...")
        self.deliveries = self.generate_deliveries()

        print("⏱️  Calculating time limits...")
        self.calculate_time_limits()

        print(f"✅ Generated {len(self.deliveries)} deliveries")

    def generate_distribution(self):
        mu_list = [peak * 60 * self.hours for peak in self.peaks]
        x = np.linspace(0, 60 * self.hours, 60 * self.hours)
        y_combined = np.zeros_like(x)
        for mu in mu_list:
            y_combined += (1 / (self.sigma * np.sqrt(2 * np.pi))) * \
                         np.exp(-0.5 * ((x - mu) / self.sigma) ** 2)
        loads = [int((self.load * (self.sigma * np.sqrt(2 * np.pi))) * y)
                for y in y_combined]
        return y_combined, loads

    def precompute_all_paths(self):
        valid_nodes = list(range(self.num_hotspots))
        total_pairs = len(valid_nodes) * (len(valid_nodes) - 1)

        with tqdm(total=total_pairs, desc="Pre-computing paths") as pbar:
            for start in valid_nodes:
                for end in valid_nodes:
                    if start != end:
                        travel_time, path = astar_shortest_path(
                            adjacency=self.adjacency_matrix,
                            start=start,
                            goal=end,
                            node_positions=self.node_positions,
                            weight_type='time'
                        )
                        self.precomputed_paths[(start, end)] = (travel_time, path)
                        if travel_time <= self.max_delivery_time:
                            self.valid_pairs.append((start, end))
                        pbar.update(1)

        print(f"✓ Pre-computed {len(self.precomputed_paths)} shortest paths")
        print(f"✓ Found {len(self.valid_pairs)} valid delivery pairs (within {self.max_delivery_time}s limit)")

        if not self.valid_pairs:
            raise ValueError(f"No valid delivery pairs found within {self.max_delivery_time}s time limit!")

    def calculate_shortest_path(self, start, end):
        if start == end: return [start]
        if (start, end) not in self.precomputed_paths:
            print(f"⚠️ WARNING: Path ({start}, {end}) not found in pre-computed paths!")
            return [start, end]
        travel_time, path = self.precomputed_paths[(start, end)]
        return path

    def get_travel_time(self, start, end):
        if start == end: return 0
        if (start, end) not in self.precomputed_paths:
            print(f"⚠️ WARNING: Path ({start}, {end}) not found in pre-computed paths!")
            return float('inf')
        travel_time, path = self.precomputed_paths[(start, end)]
        return travel_time

    def calculate_time_limits(self):
        for delivery in self.deliveries:
            travel_time = self.get_travel_time(delivery.start_node, delivery.end_node)
            required_time = travel_time * delivery.TIME_TOLERANCE_FACTOR
            for interval in INTERVALS:
                if required_time <= interval:
                    delivery.time_limit = interval
                    break
            else:
                delivery.time_limit = float('inf')

    def generate_deliveries(self):
        deliveries = []
        total_deliveries = sum(self.loads)
        if not self.valid_pairs:
            raise ValueError("No valid delivery pairs available!")

        with tqdm(total=total_deliveries, desc="Creating deliveries") as pbar:
            for j in range(60 * self.hours):
                for _ in range(self.loads[j]):
                    start, end = random.choice(self.valid_pairs)
                    travel_time, shortest_path = self.precomputed_paths[(start, end)]
                    time = random.randint(j * 60, (j + 1) * 60)
                    delivery = Delivery(
                        start, end, time,
                        time_tolerance_factor=self.time_tolerance_factor,
                        shortest_path=shortest_path
                    )
                    deliveries.append(delivery)
                    pbar.update(1)
        return deliveries

    def reset_deliveries(self):
        """Reset all deliveries to their initial state."""
        Delivery.id_counter = 0 # Reset the global ID counter
        for d in self.deliveries:
            d.reset()
            # Re-assign IDs to be consistent
            d.id = Delivery.id_counter
            Delivery.id_counter += 1

#### 4. Load Graph Data

(This is UNCHANGED from the original file)

In [131]:
import random

def inspect_random_edges(adjacency, n_samples=10):
    """
    Randomly print n_samples of edges from the adjacency dict.
    Helps verify distance/time units and detect corrupted entries.
    """
    all_edges = []
    for u, neighbors in adjacency.items():
        for v, dist, time in neighbors:
            if u < v:  # avoid duplicates since it's undirected
                all_edges.append((u, v, dist, time))

    if not all_edges:
        print("⚠️ No edges found in adjacency.")
        return

    print(f"Inspecting {min(n_samples, len(all_edges))} random edges:\n")
    for (u, v, dist, time) in random.sample(all_edges, min(n_samples, len(all_edges))):
        print(f"  Edge {u:>4} ↔ {v:<4} | Distance: {dist:.3f} | Time: {time:.3f} sec")

    print("\nℹ️ Note: If time values seem ~60× too large or too small, "
          "check whether your raw 'time' edge attribute is already in seconds or minutes.")


In [132]:
print("\n[1] Loading graph and census data...")
try:
    census_df = gpd.read_file(data_dir + "/Census Data/census_tract_data.geojson")
    num_hotspots = len(census_df.index)
    print(f"    ✓ Census tracts loaded: {num_hotspots} tracts (hotspots)")

    # --- CORRECTED FILE PATHS ---
    # Added .xml to the end based on your screenshot
    umst_path = data_dir + "/UMST Graph/graphs/umst_graph.graphml.xml"
    mst_path = data_dir + "/UMST Graph/graphs/mst_graph.graphml.xml"
    hotspot_path = data_dir + "/UMST Graph/graphs/gh_hotspot_graph.graphml.xml"

    umst_graph = nx.read_graphml(umst_path)
    print(f"    ✓ UMST Graph loaded: {umst_graph.number_of_nodes()} nodes, {umst_graph.number_of_edges()} edges")

    # --- 5. Build Adjacency & Position Dictionaries ---
    # (This section is the same as before)
    print("\n[2] Building node mappings and adjacency...")
    nodes = sorted(umst_graph.nodes())
    index_to_tract = {idx: node for idx, node in enumerate(nodes)}
    tract_to_index = {node: idx for idx, node in enumerate(nodes)}

    adjacency_matrix = build_adjacency_matrix(umst_graph, tract_to_index)
    inspect_random_edges(adjacency_matrix, n_samples=10)

    node_positions = {}
    for geo_id, data in umst_graph.nodes(data=True):
        idx = tract_to_index[geo_id]
        lat = float(data.get('lat', 0))
        lon = float(data.get('lon', 0))
        node_positions[idx] = (lat, lon)
    print("    ✓ Node position dictionary built.")

    # --- 6. Instantiate the DeliveryList ---
    # (This section is the same as before)
    print("\n[3] Generating all delivery tasks for the simulation...")
    random.seed(42) # Use a fixed seed for reproducibility
    np.random.seed(42)

    deliverylist = DeliveryList(
        max_delivery_time=1800, # all random deliveries will be under this
        load= 400,
        time_tolerance_factor=2.0,
        hours=1,
        peaks=[0.25, 0.75],
        sigma=10,
        num_hotspots=num_hotspots,
        adjacency_matrix=adjacency_matrix,
        node_positions=node_positions
    )

except Exception as e:
    print(f"\n--- !! ERROR !! ---")
    print(f"Could not load graph data or build DeliveryList.")
    print(f"Make sure your Google Drive is mounted and the path is correct:")
    print(f"PATH: {data_dir}")
    print(f"Error: {e}")


[1] Loading graph and census data...
    ✓ Census tracts loaded: 26 tracts (hotspots)
    ✓ UMST Graph loaded: 26 nodes, 47 edges

[2] Building node mappings and adjacency...
✓ Adjacency built: 26 nodes with 47 edges
Inspecting 10 random edges:

  Edge   19 ↔ 23   | Distance: 1.475 | Time: 189.000 sec
  Edge   11 ↔ 24   | Distance: 1.885 | Time: 451.000 sec
  Edge   16 ↔ 20   | Distance: 4.878 | Time: 443.000 sec
  Edge    0 ↔ 3    | Distance: 0.967 | Time: 156.000 sec
  Edge   15 ↔ 16   | Distance: 3.461 | Time: 204.000 sec
  Edge   20 ↔ 21   | Distance: 2.416 | Time: 360.000 sec
  Edge    1 ↔ 2    | Distance: 1.711 | Time: 234.000 sec
  Edge    5 ↔ 6    | Distance: 0.707 | Time: 157.000 sec
  Edge    8 ↔ 9    | Distance: 1.212 | Time: 168.000 sec
  Edge    7 ↔ 15   | Distance: 1.373 | Time: 156.000 sec

ℹ️ Note: If time values seem ~60× too large or too small, check whether your raw 'time' edge attribute is already in seconds or minutes.
    ✓ Node position dictionary built.

[3] Ge

Pre-computing paths: 100%|██████████| 650/650 [00:00<00:00, 33315.79it/s]


✓ Pre-computed 650 shortest paths
✓ Found 636 valid delivery pairs (within 1800s limit)
📦 Generating deliveries with pre-computed paths...


Creating deliveries: 100%|██████████| 18498/18498 [00:00<00:00, 212860.70it/s]

⏱️  Calculating time limits...
✅ Generated 18498 deliveries


In [133]:
import random

def create_delivery_subsets(full_deliverylist, train_frac=0.15, seed=42):
    """
    Splits DeliveryList.deliveries into train/test subsets (by IDs).
    Returns (train_list, test_list) of Delivery objects.
    """
    random.seed(seed)
    all_deliveries = full_deliverylist.deliveries
    n_total = len(all_deliveries)
    n_train = int(train_frac * n_total)

    indices = list(range(n_total))
    random.shuffle(indices)
    train_idx = set(indices[:n_train])

    train_deliveries = [all_deliveries[i] for i in train_idx]
    test_deliveries  = [all_deliveries[i] for i in range(n_total) if i not in train_idx]

    print(f"✓ Split deliveries: {len(train_deliveries)} train, {len(test_deliveries)} test (of {n_total})")
    # Debug sample
    print("  Example train IDs:", [d.id for d in train_deliveries[:5]])
    print("  Example test  IDs:", [d.id for d in test_deliveries[:5]])
    return train_deliveries, test_deliveries

train_deliveries, test_deliveries = create_delivery_subsets(deliverylist, train_frac=0.15)

✓ Split deliveries: 2774 train, 15724 test (of 18498)
  Example train IDs: [8194, 7, 8200, 8203, 8205]
  Example test  IDs: [0, 1, 2, 3, 4]


In [134]:
class DeliveryListSubset:
    """Wraps a subset of Delivery objects for quick experiments."""
    def __init__(self, deliveries, parent_deliverylist):
        self.deliveries = deliveries
        self.adjacency_matrix = parent_deliverylist.adjacency_matrix
        self.node_positions   = parent_deliverylist.node_positions

    def reset_deliveries(self):
        for d in self.deliveries:
            d.reset()

In [135]:
train_dl = DeliveryListSubset(train_deliveries, deliverylist)
test_dl  = DeliveryListSubset(test_deliveries, deliverylist)
print(f"Train subset has {len(train_dl.deliveries)} deliveries.")

Train subset has 2774 deliveries.


In [136]:
# --- Minimal DeliveryEnvShared + SharedDQNAgent implementations ---
# Drop this cell into your notebook (after Delivery & DeliveryList definitions)

import numpy as np
import random
from collections import deque
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class DeliveryEnvShared:
    """
    Simplified shared-environment for MADDPG-style evaluation used by run_rl_v7().
    - deliverylist: DeliveryList instance (with .deliveries list of Delivery objects)
    - adjacency: adjacency matrix (NxN numpy array or list-of-lists) with edge weights (km)
    - node_positions: dict or list mapping node index -> (x,y) (only for _dist)
    - n_agents: number of vehicles
    - capacity: integer per vehicle
    - sim_duration: maximum sim time (seconds / steps)
    Notes:
      - Actions are discrete integers = target node index.
      - Each step increments internal time by 1 (one step ~ 1s). Use same units as Delivery.start_time.
      - env.agents is a list of dicts with at least a 'node' field (current node).
      - env.reset(seed=...) returns an observation array shape (n_agents, obs_dim).
      - env.step(actions) returns obs, rew, dones, info similar to gym.
    """
# === REPLACE THIS FUNCTION ===
    def __init__(self, deliverylist, adjacency, node_positions,
                 n_agents=None, capacity=2, sim_duration=4500):
        self.deliverylist = deliverylist

        # --- FIX: handle both dict and numpy adjacency formats safely ---
        if isinstance(adjacency, dict):
            self.adjacency = adjacency
            self.N = len(adjacency)
        else:
            self.adjacency = np.array(adjacency)
            self.N = self.adjacency.shape[0]

        self.node_positions = node_positions
        self.n_agents = n_agents or 1
        self.capacity = capacity
        self.sim_duration = sim_duration

        # Extract neighbors list
        self.neighbors = []
        for i in range(self.N):
            if isinstance(self.adjacency, dict):
                # adjacency[i] -> [(neighbor, dist, time), ...]
                self.neighbors.append([nbr for nbr, _, _ in self.adjacency.get(i, [])])
            else:
                self.neighbors.append(np.where(self.adjacency[i] > 0)[0].tolist())

        # Precompute shortest paths
        self._shortest_paths = {}
        self._compute_all_pairs_shortest_paths()

        # Internal state
        self.time = 0
        self.agents = None
        
        # === THIS IS THE CHANGE ===
        # We need a way to look up deliveries by ID for the _agent_obs
        # We also store the full list of deliveries
        self.deliveries = self.deliverylist.deliveries
        self._delivery_dict = {d.id: d for d in self.deliveries}
        
        # === THIS IS THE CHANGE ===
        # Update the _obs_dim
        # [node, carried, time] + [carried_dest_map] + [pickup_map]
        self._obs_dim = 6
        # === END CHANGE ===
        
        self.max_distance = max(
            np.linalg.norm(np.array(self.node_positions[i]) - np.array(self.node_positions[j]))
            for i in range(self.N) for j in range(self.N) if i != j
        )


        print(f"[DEBUG] Environment created with {self.n_agents} agents, "
              f"{len(deliverylist.deliveries)} deliveries, {self.N} nodes. "
              f"Obs dim: {self._obs_dim}")


    def _compute_all_pairs_shortest_paths(self):
        # BFS from each node (since N likely modest). Store as list of next-hop lists.
        for s in range(self.N):
            # BFS tree
            prev = [-1]*self.N
            q = deque([s])
            prev[s] = s
            while q:
                u = q.popleft()
                for v in self.neighbors[u]:
                    if prev[v] == -1:
                        prev[v] = u
                        q.append(v)
            # recover paths
            for d in range(self.N):
                if prev[d] == -1:
                    continue
                path = []
                cur = d
                while cur != s:
                    path.append(cur)
                    cur = prev[cur]
                path.append(s)
                path.reverse()
                self._shortest_paths[(s,d)] = path

    def _path(self, a, b):
        if a == b:
            return [a]
        return self._shortest_paths.get((a,b), [a])  # fallback to staying put

    def _dist(self, a, b):
        # Euclidean if positions provided, else graph-heuristic using adjacency weights
        try:
            pa = self.node_positions[a]; pb = self.node_positions[b]
            dx = pa[0] - pb[0]; dy = pa[1] - pb[1]
            return math.hypot(dx, dy)
        except Exception:
            # sum along shortest path using adjacency weights (if adjacency contains distances)
            path = self._path(a,b)
            d = 0.0
            for i in range(len(path)-1):
                d += self.adjacency[path[i], path[i+1]]
            return d

    def reset(self, seed=None):
        if seed is not None:
            random.seed(seed); np.random.seed(seed)

        # Reset global deliveries to their initial state if possible
        try:
            self.deliverylist.reset_deliveries()
        except Exception:
            pass
        
        # === THIS IS THE CHANGE ===
        # Make sure our internal references are updated
        self.deliveries = self.deliverylist.deliveries
        self._delivery_dict = {d.id: d for d in self.deliveries}
        # === END CHANGE ===

        # Clear any transient delivery flags (safety)
        for d in self.deliveries:
            if hasattr(d, '_just_picked'): delattr(d, '_just_picked')
            if hasattr(d, '_just_delivered'): delattr(d, '_just_delivered')
            if hasattr(d, '_just_delivered_by'): delattr(d, '_just_delivered_by')
            # ensure delivery bookkeeping fields exist
            d.in_transition = False
            d.completed = False
            d.successful = False
            d.current_bundle_id = None

        # Place agents: start at randomly sampled nodes (or deterministic if few nodes)
        start_nodes = list(range(self.N))
        if len(start_nodes) >= self.n_agents:
            starts = random.sample(start_nodes, self.n_agents)
        else:
            starts = [start_nodes[i % len(start_nodes)] for i in range(self.n_agents)]

        self.agents = []
        for a in range(self.n_agents):
            agent = {
                'id': a,
                'node': int(starts[a]),
                'target': int(starts[a]),
                'carried': [],              # list of delivery ids currently on this vehicle
                'distance_travelled': 0.0,
                '_last_distance': 0.0,      # initialize last distance here (fixes the TypeError)
            }
            self.agents.append(agent)

        self.time = 0

        # === THIS IS THE CHANGE ===
        # The obs array shape is now based on the new _obs_dim
        obs = np.zeros((self.n_agents, self._obs_dim), dtype=float)
        # === END CHANGE ===
        
        for i, ag in enumerate(self.agents):
            obs[i] = self._agent_obs(i)
        return obs


    def _agent_obs(self, agent_idx):
        ag = self.agents[agent_idx]
        
        # === 1. Self-State (3 dims) ===
        node_norm = ag['node'] / max(1, self.N - 1)
        carried_norm = len(ag['carried']) / max(1, self.capacity)
        time_norm = self.time / max(1, self.sim_duration)
        
        # === 2. Nearest Pickup Distance ===
        nearest_pickup_dist = np.inf
        for d in self.deliveries:
            if (not d.in_transition) and (not d.completed) and (self.time >= d.start_time):
                # approximate distance using Euclidean node positions
                start_pos = self.node_positions[d.start_node]
                agent_pos = self.node_positions[ag['node']]
                dist = np.linalg.norm(np.array(start_pos) - np.array(agent_pos))
                nearest_pickup_dist = min(nearest_pickup_dist, dist)
        if np.isinf(nearest_pickup_dist):
            nearest_pickup_dist = 0.0
        nearest_pickup_dist_norm = nearest_pickup_dist / (self.max_distance + 1e-8)

        # === 3. Nearest Dropoff Distance ===
        nearest_dropoff_dist = 0.0
        if ag['carried']:
            dists = []
            for delivery_id in ag['carried']:
                if delivery_id in self._delivery_dict:
                    dest_node = self._delivery_dict[delivery_id].end_node
                    dest_pos = self.node_positions[dest_node]
                    agent_pos = self.node_positions[ag['node']]
                    dist = np.linalg.norm(np.array(dest_pos) - np.array(agent_pos))
                    dists.append(dist)
            if dists:
                nearest_dropoff_dist = min(dists)
        nearest_dropoff_dist_norm = nearest_dropoff_dist / (self.max_distance + 1e-8)

        # === 4. Pending Deliveries (global load signal) ===
        total_active = sum((not d.completed) for d in self.deliveries)
        pending_deliveries_norm = total_active / max(1, len(self.deliveries))

        # === Final concatenation ===
        obs = np.array([
            node_norm,
            carried_norm,
            time_norm,
            nearest_pickup_dist_norm,
            nearest_dropoff_dist_norm,
            pending_deliveries_norm
        ], dtype=float)
        
        return obs


    def step(self, actions):
        """
        actions: array-like of ints of length n_agents (target node indices)
        Returns: obs (n_agents, obs_dim), rewards (n_agents,), dones (n_agents,), info (dict)
        """

        actions = np.asarray(actions, dtype=int)
        actions = np.clip(actions, 0, self.N-1)  # no movement restriction beyond valid indices

        pickups = 0
        deliveries_done = 0

        # Initialize reward array
        reward = np.zeros(self.n_agents, dtype=float)

        # --- Apply actions ---
        for i, a in enumerate(self.agents):
            a['target'] = int(actions[i])

        # --- Move each agent one hop towards its target ---
        for a in self.agents:
            cur = a['node']
            tgt = a['target']
            if cur != tgt:
                path = self._path(cur, tgt)
                nxt = path[1] if len(path) >= 2 else cur
                d = self._dist(cur, nxt)
                a['distance_travelled'] += d
                a['node'] = int(nxt)

        # --- Advance time ---
        self.time += 1
        done_global = self.time > self.sim_duration

        # --- Pickup logic ---
        for a in self.agents:
            if len(a['carried']) >= self.capacity:
                continue
            for d in self.deliveries:
                if (not d.in_transition) and (not d.completed) and (d.start_node == a['node']):
                    if self.time >= d.start_time:
                        d.in_transition = True
                        d.current_node = a['node']
                        d.path_index = 0
                        a['carried'].append(d.id)
                        d.current_bundle_id = a['id']
                        d._just_picked = True  # for reward
                        pickups += 1
                        if len(a['carried']) >= self.capacity:
                            break

        # --- Delivery logic ---
        for d in self.deliveries:
            if d.in_transition and (not d.completed):
                carrier = next((a for a in self.agents if d.id in a['carried']), None)
                if carrier is None:
                    continue

                # Deliver
                if carrier['node'] == d.end_node:
                    d.completed = True
                    d.successful = True if (self.time <= d.time_limit if d.time_limit else True) else False
                    d.end_time = self.time
                    d.in_transition = False
                    d.current_node = d.end_node
                    carrier['carried'].remove(d.id)
                    d._just_delivered = True
                    d._just_delivered_by = carrier['id']
                    deliveries_done += 1
                    d.distance_traveled = self._dist(d.start_node, d.end_node)
                else:
                    d.current_node = carrier['node']

        # --- Compute total reward ---
        reward = self._compute_rewards(reward)

        # --- Observations ---
        obs = np.zeros((self.n_agents, self._obs_dim), dtype=float)
        for i in range(self.n_agents):
            obs[i] = self._agent_obs(i)

        dones = np.array([done_global]*self.n_agents)
        info = {
            'pickups': pickups,
            'deliveries': deliveries_done,
            'vehicle_distance': sum(a['distance_travelled'] for a in self.agents)
        }

        return obs, reward, dones, info
    
    def _compute_rewards(self, reward):
        delivered_counts = {a['id']: 0 for a in self.agents}

        # --- DEBUG counters ---
        self.debug_stats = getattr(self, 'debug_stats', {
            'idle_penalty': 0,
            'move_bonus': 0,
            'pickup_bonus': 0,
            'delivery_bonus': 0
        })

        for a in self.agents:
            last_dist = a.get('_last_distance', 0.0)
            cur_dist = a['distance_travelled']
            move_delta = cur_dist - last_dist
            a['_last_distance'] = cur_dist

            reward[a['id']] -= 0.0005  # time cost

            if move_delta > 0:
                reward[a['id']] += 0.001
                self.debug_stats['move_bonus'] += 1   # DEBUG

            if move_delta == 0.0 and any(
                (not d.in_transition) and (not d.completed) and (self.time >= d.start_time)
                for d in self.deliveries
            ):
                reward[a['id']] -= 0.005
                self.debug_stats['idle_penalty'] += 1   # DEBUG

        for d in self.deliveries:
            if getattr(d, '_just_picked', False):
                reward[d.current_bundle_id] += 15.0
                self.debug_stats['pickup_bonus'] += 1   # DEBUG
                d._just_picked = False

            if getattr(d, '_just_delivered', False):
                carrier_id = d._just_delivered_by
                if d.successful:
                    reward[carrier_id] += 100.0
                else:
                    reward[carrier_id] += 5.0
                self.debug_stats['delivery_bonus'] += 1   # DEBUG
                delivered_counts[carrier_id] += 1
                d._just_delivered = False
                d._just_delivered_by = None

        for a in self.agents:
            k = delivered_counts[a['id']]
            if k > 1:
                reward[a['id']] += 2.0 * (k - 1)

        if np.any(reward > 1.0):
            reward = np.tanh(reward / 20.0)
        reward = np.clip(reward, -1.0, 1.0)
        return reward


# --------------------
# Shared minimal DQN-style policy wrapper (vectorized)
# --------------------
class SimpleQNet(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.linear1 = nn.Linear(obs_dim, hidden)
        self.linear2 = nn.Linear(hidden, hidden)
        self.head = nn.Linear(hidden, n_actions)

    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        return self.head(x)

class SharedDQNAgent:
    """
    Minimal shared DQN-style agent that exposes `act(obs_batch, epsilon=0.0)`.
    - obs_batch: numpy array (B, obs_dim) or torch tensor
    - n_actions: typically set to number of nodes (agent chooses which node to go to)
    - This class does NOT implement training loops or replay buffer here;
      it is only an evaluation-time policy wrapper.
    """
    def __init__(self, obs_dim, n_actions, device=None, hidden=64):
        self.obs_dim = obs_dim
        self.n_actions = n_actions
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.net = SimpleQNet(obs_dim, n_actions, hidden).to(self.device)
        # random init; user can load state_dict later if they have pretrained weights
        self.net.eval()

    def act(self, obs_batch, epsilon=0.0):
        """
        obs_batch: numpy array shape (B, obs_dim)
        returns: numpy array of ints shape (B,) with chosen action indices
        """
        if isinstance(obs_batch, np.ndarray):
            x = torch.from_numpy(obs_batch).float().to(self.device)
        else:
            x = obs_batch.float().to(self.device)

        # epsilon-greedy
        if epsilon > 0.0 and np.random.rand() < epsilon:
            # random actions
            B = x.shape[0]
            return np.random.randint(0, self.n_actions, size=B, dtype=int)

        with torch.no_grad():
            q = self.net(x)  # (B, n_actions)
            actions = q.argmax(dim=1).cpu().numpy().astype(int)
        return actions

    def predict(self, obs_batch):
        "Return Q-values for diagnostic purposes."
        if isinstance(obs_batch, np.ndarray):
            x = torch.from_numpy(obs_batch).float().to(self.device)
        else:
            x = obs_batch.float().to(self.device)
        with torch.no_grad():
            q = self.net(x).cpu().numpy()
        return q

    def load(self, path):
        "Load state_dict (torch) into the internal net."
        st = torch.load(path, map_location=self.device)
        self.net.load_state_dict(st)
        self.net.eval()


In [137]:
import time

start = time.time()
obs = env.reset(seed=0)
done = np.array([False]*env.n_agents)
steps = 0
while not done.all() and steps < 200:
    actions = np.random.randint(0, env.N, size=env.n_agents)
    obs, rew, done, info = env.step(actions)
    steps += 1
end = time.time()

print(f"Episode finished in {steps} steps, took {end-start:.2f}s.")
print("Info summary:", info)


Episode finished in 200 steps, took 0.69s.
Info summary: {'pickups': 0, 'deliveries': 0, 'vehicle_distance': 7.846275480084916}


In [138]:
# ===== MADDPG training for your DeliveryEnvShared =====
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import os
from collections import deque, namedtuple

# ---------- Hyperparameters (tweakable) ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_AGENTS = 4                      # must match your env instantiation
OBS_DIM = 6 #env._obs_dim              # from your env (3)
N_NODES = env.N                     # number of discrete actions (nodes)
ACT_DIM = 1                         # continuous scalar per agent (we'll map -> node index)
HIDDEN = 128
BUFFER_SIZE = 200000
BATCH_SIZE = 256
GAMMA = 0.99
TAU = 0.01
# new local movement hyperparam
# LOCAL_MAX_FACTOR = 4.0  # <-- This is no longer used
LR_ACTOR = 3e-4
LR_CRITIC = 3e-4
MAX_EPISODES = 1500                 # start moderate; increase if you have time
MAX_STEPS = 1800                    # per episode truncation (you used 200 earlier)
START_TRAIN_AFTER = 2000            # number of transitions before learning begins
TRAIN_EVERY = 1                    # learn every N environment steps
N_UPDATES = 2                      # gradient steps per learning

# --- Exploration noise schedule ---
noise_start = 0.5   # initial exploration
noise_end = 0.05    # minimum
noise_decay_episodes = 1200

def get_noise_scale(ep):
    ratio = min(1.0, ep / noise_decay_episodes)
    return noise_start - (noise_start - noise_end) * ratio

noise_scale = get_noise_scale(0)


SAVE_DIR = output_dir
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------- Replay buffer (joint experience) ----------
Transition = namedtuple("Transition", ["obs", "actions", "rewards", "next_obs", "dones"])

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, *args):
        self.buffer.append(Transition(*args))
    def sample(self, batch_size):
        batch = random.sample(list(self.buffer), batch_size)
        # Convert to arrays with shapes: (B, num_agents, obs_dim / 1 etc.)
        obs = np.stack([b.obs for b in batch], axis=0)           # (B, A, obs_dim)
        actions = np.stack([b.actions for b in batch], axis=0)    # (B, A)
        rewards = np.stack([b.rewards for b in batch], axis=0)    # (B, A)
        next_obs = np.stack([b.next_obs for b in batch], axis=0)  # (B, A, obs_dim)
        dones = np.stack([b.dones for b in batch], axis=0)      # (B, A)
        return obs, actions, rewards, next_obs, dones
    def __len__(self):
        return len(self.buffer)

# ---------- Networks ----------
def mlp(in_dim, out_dim, hidden=HIDDEN):
    return nn.Sequential(
        nn.Linear(in_dim, hidden),
        nn.ReLU(),
        nn.Linear(hidden, hidden),
        nn.ReLU(),
        nn.Linear(hidden, out_dim)
    )

# modify Actor.forward
class Actor(nn.Module):
    def __init__(self, obs_dim, hidden=HIDDEN):
        super().__init__()
        self.net = mlp(obs_dim, ACT_DIM, hidden)
    def forward(self, x):
        # bound output to (-1, 1) so mapping is stable
        return torch.tanh(self.net(x))


class Critic(nn.Module):
    def __init__(self, joint_obs_dim, joint_act_dim, hidden=HIDDEN):
        super().__init__()
        self.net = mlp(joint_obs_dim + joint_act_dim, 1, hidden)
    def forward(self, joint_obs, joint_actions):
        # joint_obs: (B, A*obs_dim), joint_actions: (B, A*ACT_DIM)
        x = torch.cat([joint_obs, joint_actions], dim=-1)
        return self.net(x).squeeze(-1)  # (B,)
    
# ---------- Helper mapping continuous -> discrete (GLOBAL node index, no movement restriction) ----------
def cont_to_node_index(x_cont):
    """
    Map a continuous scalar (float) to a global node index in [0, N_NODES-1].
    Uses tanh to keep it bounded, then scales to integer index.
    """
    s = (np.tanh(float(x_cont)) + 1.0) / 2.0   # squashes to [0, 1]
    idx = int(np.clip(round(s * (N_NODES - 1)), 0, N_NODES - 1))
    return idx

def batch_cont_to_nodes(x_cont_batch):
    """
    Map a batch of continuous scalars to global node indices [0, N_NODES-1].
    Accepts (A,) or (B, A) numpy arrays.
    """
    arr = np.array(x_cont_batch, dtype=float)
    s = (np.tanh(arr) + 1.0) / 2.0
    idxs = np.round(s * (N_NODES - 1)).astype(int)
    return np.clip(idxs, 0, N_NODES - 1)


# ---------- MADDPG agent containers ----------
class MADDPG:
    def __init__(self, num_agents, obs_dim, act_dim):
        self.num_agents = num_agents
        self.obs_dim = obs_dim
        self.act_dim = act_dim

        # per-agent actor and target
        self.actors = [Actor(obs_dim).to(device) for _ in range(num_agents)]
        self.targets_actor = [Actor(obs_dim).to(device) for _ in range(num_agents)]
        for a, t in zip(self.actors, self.targets_actor):
            t.load_state_dict(a.state_dict())

        # per-agent critic and target (each critic observes joint obs+joint actions)
        joint_obs_dim = num_agents * obs_dim
        joint_act_dim = num_agents * act_dim
        self.critics = [Critic(joint_obs_dim, joint_act_dim).to(device) for _ in range(num_agents)]
        self.targets_critic = [Critic(joint_obs_dim, joint_act_dim).to(device) for _ in range(num_agents)]
        for c, tc in zip(self.critics, self.targets_critic):
            tc.load_state_dict(c.state_dict())

        # optimizers
        self.opt_actors = [optim.Adam(a.parameters(), lr=LR_ACTOR) for a in self.actors]
        self.opt_critics = [optim.Adam(c.parameters(), lr=LR_CRITIC) for c in self.critics]

    def act(self, obs_batch, noise_scale=0.0):
        # obs_batch: np array (B, A, obs_dim) or (A, obs_dim) for single-step
        single = False
        if obs_batch.ndim == 2:  # (A, obs_dim)
            obs_batch = np.expand_dims(obs_batch, axis=0)  # (1, A, obs_dim)
            single = True
        B = obs_batch.shape[0]
        obs_t = torch.tensor(obs_batch, dtype=torch.float32, device=device)  # (B, A, obs_dim)
        actions = []
        for i in range(self.num_agents):
            xi = obs_t[:, i, :]  # (B, obs_dim)
            with torch.no_grad():
                out = self.actors[i](xi).cpu().numpy()  # (B, ACT_DIM)
            # add noise
            out = out + np.random.randn(*out.shape) * noise_scale
            actions.append(out.reshape(B, -1))
        # actions list -> (B, A*ACT_DIM)
        acts = np.concatenate(actions, axis=1)
        # map each agent's scalar to nearest node index
        # reshape to (B, A)
        acts_per_agent = acts.reshape(B, self.num_agents)
        discrete = batch_cont_to_nodes(acts_per_agent)
        if single:
            return discrete[0], acts_per_agent[0]  # (A,), (A,) continuous before rounding
        return discrete, acts_per_agent

    def target_act_cont(self, next_obs_batch):
        # returns continuous actions from target actors (B, A)
        B = next_obs_batch.shape[0] if next_obs_batch.ndim == 3 else 1
        obs_t = torch.tensor(next_obs_batch, dtype=torch.float32, device=device)
        conts = []
        for i in range(self.num_agents):
            xi = obs_t[:, i, :]
            with torch.no_grad():
                out = self.targets_actor[i](xi).cpu().numpy()
            conts.append(out.reshape(B, -1))
        conts = np.concatenate(conts, axis=1)  # (B, A*ACT_DIM)
        return conts.reshape(B, self.num_agents)

    def soft_update(self):
        for i in range(self.num_agents):
            for p, tp in zip(self.actors[i].parameters(), self.targets_actor[i].parameters()):
                tp.data.copy_(tp.data * (1.0 - TAU) + p.data * TAU)
            for p, tp in zip(self.critics[i].parameters(), self.targets_critic[i].parameters()):
                tp.data.copy_(tp.data * (1.0 - TAU) + p.data * TAU)

# ---------- Initialize MADDPG and replay ----------
maddpg = MADDPG(NUM_AGENTS, OBS_DIM, ACT_DIM)
replay = ReplayBuffer(BUFFER_SIZE)

# ---------- Utility: convert numpy batch to torch joint tensors ----------
def to_joint_tensors(obs_batch, cont_actions_batch):
    # obs_batch: (B, A, obs_dim) -> flatten to (B, A*obs_dim)
    B = obs_batch.shape[0]
    jobs = torch.tensor(obs_batch.reshape(B, -1), dtype=torch.float32, device=device)
    jacts = torch.tensor(cont_actions_batch.reshape(B, -1), dtype=torch.float32, device=device)
    return jobs, jacts

# ---------- Training loop ----------
total_steps = 0

# Safe version: filters noisy debug prints without recursion
import builtins

# Save the original print only once
if not hasattr(builtins, "_original_print"):
    builtins._original_print = builtins.print

def print_filtered(*args, **kwargs):
    if args and isinstance(args[0], str) and args[0].startswith("[DEBUG] Environment created"):
        return
    builtins._original_print(*args, **kwargs)

builtins.print = print_filtered




print("Starting MADDPG training on train_dl ...")

# ---------- Resume from checkpoint if available ----------
resume_from = "maddpg_ep60.pt" #if you want to restart# e.g., "maddpg_ep60.pt" if you want to resume later
if resume_from:
    checkpoint = torch.load(os.path.join(SAVE_DIR, resume_from), map_location=device)
    for a, sd in zip(maddpg.actors, checkpoint['actors']):
        a.load_state_dict(sd)
    for c, sd in zip(maddpg.critics, checkpoint['critics']):
        c.load_state_dict(sd)
    print(f"✅ Resumed from checkpoint {resume_from}")
else:
    print("Starting fresh training...")


from collections import deque
recent_rewards = deque(maxlen=10)  # track last 10 episode means





for ep in range(1, MAX_EPISODES + 1):
    # create fresh env for each episode
    env_train = DeliveryEnvShared(
        train_dl,
        adjacency_matrix,
        node_positions,
        n_agents=NUM_AGENTS,
        capacity=2,
        sim_duration=MAX_STEPS
    )
    noise_scale = get_noise_scale(ep)
    
    obs = env_train.reset(seed=ep)
    ep_reward = np.zeros(NUM_AGENTS)
    done = np.array([False]*NUM_AGENTS)
    steps = 0
    
    # diagnostics
    ep_pickups = 0
    ep_deliveries = 0
    ep_distance = 0.0

    # ==================================================================
    # ===== START: MODIFIED TRAINING LOOP (BUG FIX 1 APPLIED) ======
    # ==================================================================
    while (not done.all()) and steps < MAX_STEPS:
        
        # Get both discrete actions (for env) and continuous actions (for replay)
        # discrete_actions shape: (A,) -> [node_idx_0, node_idx_1, ...]
        # cont_actions shape: (A,) -> [scalar_0, scalar_1, ...]
        discrete_actions, cont_actions = maddpg.act(obs, noise_scale=noise_scale)
        cont_actions_for_replay = np.tanh(cont_actions)
        # --- DELETED BLOCK ---
        # The entire "LOCAL NEIGHBOR MAPPING" block has been removed.
        # We now use the global discrete_actions directly.
        
        # Step the environment using the globally mapped DISCRETE node indices
        next_obs, rew, done, info = env_train.step(discrete_actions)
        
        if next_obs is None or rew is None or done is None or obs is None:
            print("⚠️ NoneType detected:",
                  f"obs={obs is None}, rew={rew is None}, done={done is None}, next_obs={next_obs is None}")

        # Push the CONTINUOUS actions to the replay buffer
        # This fixes the mismatch: the critic now learns Q(s, cont_action)
        # by observing the reward 'rew' that resulted from step(map(cont_action)).
        replay.push(obs.copy(), cont_actions_for_replay.copy(), rew.copy(), next_obs.copy(), done.copy())
        
        obs = next_obs
        ep_reward += rew
        steps += 1
        total_steps += 1

        # collect episode diagnostics
        ep_pickups += info.get('pickups', 0)
        ep_deliveries += info.get('deliveries', 0)
        ep_distance += info.get('vehicle_distance', 0)

        # learning
        if len(replay) > BATCH_SIZE and total_steps > START_TRAIN_AFTER and total_steps % TRAIN_EVERY == 0:
            for _ in range(N_UPDATES):
                obs_b, acts_b, rews_b, next_obs_b, dones_b = replay.sample(BATCH_SIZE)
                jobs, jacts = to_joint_tensors(obs_b, acts_b)

                B = next_obs_b.shape[0]
                jnext_obs = torch.tensor(next_obs_b.reshape(B, -1), dtype=torch.float32, device=device)
                next_cont_actions = maddpg.target_act_cont(next_obs_b)
                jnext_acts = torch.tensor(next_cont_actions.reshape(B, -1), dtype=torch.float32, device=device)

                for agent_idx in range(NUM_AGENTS):
                    critic = maddpg.critics[agent_idx]
                    target_critic = maddpg.targets_critic[agent_idx]
                    opt_c = maddpg.opt_critics[agent_idx]

                    r = torch.tensor(rews_b[:, agent_idx], dtype=torch.float32, device=device)
                    d = torch.tensor(dones_b[:, agent_idx].astype(float), dtype=torch.float32, device=device)

                    with torch.no_grad():
                        q_next = target_critic(jnext_obs, jnext_acts.to(device))
                        q_target = r + GAMMA * (1.0 - d) * q_next

                    q_val = critic(jobs, jacts)
                    critic_loss = nn.MSELoss()(q_val, q_target.detach())
                    opt_c.zero_grad()
                    critic_loss.backward()
                    torch.nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
                    opt_c.step()

                    actor = maddpg.actors[agent_idx]
                    opt_a = maddpg.opt_actors[agent_idx]
                    obs_tensor = torch.tensor(
                        obs_b.reshape(BATCH_SIZE, NUM_AGENTS, OBS_DIM)[:, agent_idx, :],
                        dtype=torch.float32,
                        device=device
                    )
                    cont_act_pred = actor(obs_tensor)
                    acts_pred = torch.tensor(acts_b, dtype=torch.float32, device=device)
                    acts_pred[:, agent_idx] = cont_act_pred.squeeze(-1)
                    jacts_pred = acts_pred.reshape(BATCH_SIZE, -1)
                    q_pred = critic(jobs, jacts_pred)
                    actor_loss = -q_pred.mean()

                    opt_a.zero_grad()
                    actor_loss.backward()
                    opt_a.step()

                maddpg.soft_update()
    
    # ==================================================================
    # ===== END: MODIFIED TRAINING LOOP ================================
    # ==================================================================

    # end of episode

    # average reward across agents
    mean_rew = ep_reward.mean()
    recent_rewards.append(mean_rew)
    avg_pickups = ep_pickups / max(1, steps)
    avg_deliveries = ep_deliveries / max(1, steps)
    avg_distance = ep_distance / max(1, steps)

    if ep % 5 == 0 or ep == 1:
        print(f"Ep {ep:03d}/{MAX_EPISODES} | meanR {mean_rew:+.3f} | "
              f"pickups/step {avg_pickups:.3f} | deliveries/step {avg_deliveries:.3f} | "
              f"dist/step {avg_distance:.3f} | noise {noise_scale:.3f} | replay {len(replay)}")
    if ep % 5 == 0:
        print(f"pickups={ep_pickups}, deliveries={ep_deliveries}, dist={ep_distance:.1f}")

    if hasattr(env_train, "debug_stats"):
        dbg = env_train.debug_stats
        print(f"[Reward breakdown] move={dbg['move_bonus']}, idle={dbg['idle_penalty']}, "
            f"pickup={dbg['pickup_bonus']}, delivery={dbg['delivery_bonus']}")


    if ep % 10 == 0:
       print(f"[Debug] Avg reward last 10 eps: {np.mean(recent_rewards):.3f}")

    # occasional checkpoint
    if ep % 20 == 0:
        torch.save({
            'actors': [a.state_dict() for a in maddpg.actors],
            'critics': [c.state_dict() for c in maddpg.critics],
        }, os.path.join(SAVE_DIR, f"maddpg_ep{ep}.pt"))
        print(f"Saved checkpoint ep{ep}")


    if ep % 50 == 0:
        print(f"--- Episode {ep}/{MAX_EPISODES}, noise={noise_scale:.3f} ---")

# restore normal print
builtins.print = builtins._original_print
print = builtins.print

print("Training finished.")
torch.save({
    'actors': [a.state_dict() for a in maddpg.actors],
    'critics': [c.state_dict() for c in maddpg.critics],
}, os.path.join(SAVE_DIR, "maddpg_final.pt"))
print("Final models saved.")


Starting MADDPG training on train_dl ...
✅ Resumed from checkpoint maddpg_ep60.pt
Ep 001/1500 | meanR +11.900 | pickups/step 0.020 | deliveries/step 0.016 | dist/step 38.244 | noise 0.500 | replay 1800
[Reward breakdown] move=6775, idle=425, pickup=36, delivery=28
[Reward breakdown] move=6662, idle=537, pickup=36, delivery=28
[Reward breakdown] move=6641, idle=559, pickup=22, delivery=14
[Reward breakdown] move=6681, idle=517, pickup=15, delivery=7
Ep 005/1500 | meanR +10.492 | pickups/step 0.019 | deliveries/step 0.014 | dist/step 37.265 | noise 0.498 | replay 9000
pickups=34, deliveries=26, dist=67077.6
[Reward breakdown] move=6623, idle=576, pickup=34, delivery=26
[Reward breakdown] move=6636, idle=564, pickup=25, delivery=17
[Reward breakdown] move=6710, idle=490, pickup=53, delivery=45
[Reward breakdown] move=6703, idle=496, pickup=35, delivery=27
[Reward breakdown] move=6667, idle=533, pickup=32, delivery=24
Ep 010/1500 | meanR +7.020 | pickups/step 0.012 | deliveries/step 0.008 

KeyboardInterrupt: 

In [ ]:
print("Models will be saved to: 📂 ", os.path.abspath(SAVE_DIR))

Models will be saved to: 📂  /root/UMST_approach/Delivery_Data/Columbus_mini - RL Delivery Data/UMST Graph/maddpg_baseline


### Sanity Checks

In [ ]:
print("Number of nodes:", len(adjacency_matrix))
print("Example entry for node 0:", adjacency_matrix.get(0, []))
inspect_random_edges(adjacency_matrix, n_samples=5)

print("\nNode positions sample (first 3):")
for k in list(node_positions.keys())[:3]:
    print(k, "->", node_positions[k])


Number of nodes: 26
Example entry for node 0: [(1, 1.016, 207.0), (2, 1.829, 299.0), (3, 0.967, 156.0)]
Inspecting 5 random edges:

  Edge    4 ↔ 7    | Distance: 1.086 | Time: 156.000 sec
  Edge   19 ↔ 20   | Distance: 1.289 | Time: 234.000 sec
  Edge    2 ↔ 4    | Distance: 1.464 | Time: 261.000 sec
  Edge   11 ↔ 24   | Distance: 1.885 | Time: 451.000 sec
  Edge   17 ↔ 19   | Distance: 4.470 | Time: 372.000 sec

ℹ️ Note: If time values seem ~60× too large or too small, check whether your raw 'time' edge attribute is already in seconds or minutes.

Node positions sample (first 3):
1 -> (40.0094681290053, -83.01279606706105)
21 -> (39.973786476294, -82.99236835151807)
2 -> (40.00045262230653, -83.01230072161665)


In [ ]:
print(f"Total deliveries: {len(deliverylist.deliveries)}")
print("Sample 3 deliveries:")
for d in deliverylist.deliveries[:3]:
    print(f"ID {d.id}: {d.start_node} → {d.end_node}, "
          f"shortest_path={d.shortest_path[:5]}..., "
          f"time_limit={d.time_limit}, start_time={d.start_time}")


Total deliveries: 18498
Sample 3 deliveries:
ID 0: 4 → 16, shortest_path=[4, 8, 15, 16]..., time_limit=1200, start_time=1
ID 1: 11 → 7, shortest_path=[11, 10, 8, 7]..., time_limit=1200, start_time=15
ID 2: 9 → 4, shortest_path=[9, 8, 4]..., time_limit=900, start_time=8


In [ ]:
env_test = DeliveryEnvShared(train_dl, train_dl.adjacency_matrix, train_dl.node_positions, n_agents=1)
obs = env_test.reset(seed=0)
print("Initial obs:", obs)
print("Initial node:", env_test.agents[0]['node'])
actions = [np.random.randint(0, env_test.N)]
next_obs, rew, done, info = env_test.step(actions)
print("Next obs:", next_obs)
print("Reward:", rew)
print("Info:", info)


Initial obs: [[0.48 0.   0.  ]]
Initial node: 12
Next obs: [[4.80000000e-01 0.00000000e+00 2.22222222e-04]]
Reward: [-1.]
Info: {'pickups': 0, 'deliveries': 0, 'vehicle_distance': 0.0}


In [ ]:
env_test = DeliveryEnvShared(train_dl, train_dl.adjacency_matrix, train_dl.node_positions, n_agents=2)
obs = env_test.reset(seed=1)

total_pickups = 0
total_deliveries = 0
for step in range(200):
    actions = np.random.randint(0, env_test.N, size=env_test.n_agents)
    obs, rew, done, info = env_test.step(actions)
    total_pickups += info.get("pickups", 0)
    total_deliveries += info.get("deliveries", 0)

print(f"After 200 random steps -> pickups={total_pickups}, deliveries={total_deliveries}")


After 200 random steps -> pickups=15, deliveries=11


In [ ]:
env_test = DeliveryEnvShared(train_dl, train_dl.adjacency_matrix, train_dl.node_positions, n_agents=2)
_ = env_test.reset()
reward = np.zeros(env_test.n_agents)
for a in env_test.agents:
    a['_last_distance'] = 0
    a['distance_travelled'] = 10
reward = env_test._compute_rewards(reward)
print("Manual reward test:", reward)


Manual reward test: [-1. -1.]


In [ ]:
replay_test = ReplayBuffer(10)
dummy_obs = np.zeros((env.n_agents, env._obs_dim))
for i in range(5):
    replay_test.push(dummy_obs, np.zeros(env.n_agents), np.zeros(env.n_agents), dummy_obs, np.zeros(env.n_agents))
print("Replay size:", len(replay_test))
print("Sample batch shapes:", [x.shape for x in replay_test.sample(3)[:2]])


Replay size: 5
Sample batch shapes: [(3, 4, 3), (3, 4)]


## Results

In [42]:
import numpy as np
import torch

def evaluate_maddpg(maddpg, delivery_subset, adjacency_matrix, node_positions,
                    episodes=5, max_steps=1800, n_agents=4, capacity=2):
    """
    Evaluates trained MADDPG agents on test deliveries.
    Prints per-episode and aggregated performance metrics.
    """
    maddpg.eval_mode = True
    all_metrics = {
        'total_deliveries': 0,
        'successful_deliveries': 0,
        'failed_deliveries': 0,
        'avg_reward': [],
        'avg_distance': [],
        'avg_pickups': [],
        'vehicle_distances': [],
        'deliveries_per_agent': np.zeros(n_agents),
        'distance_per_agent': np.zeros(n_agents),
    }

    for ep in range(1, episodes + 1):
        env_eval = DeliveryEnvShared(
            delivery_subset,
            adjacency_matrix,
            node_positions,
            n_agents=n_agents,
            capacity=capacity,
            sim_duration=max_steps
        )

        obs = env_eval.reset(seed=ep)
        done = np.array([False] * n_agents)
        total_reward = np.zeros(n_agents)
        total_pickups = 0
        total_deliveries = 0
        total_distance = 0.0
        steps = 0

        while not done.all() and steps < max_steps:
            # Greedy (no noise)
            actions, _ = maddpg.act(obs, noise_scale=0.0)
            next_obs, rew, done, info = env_eval.step(actions)

            total_reward += rew
            total_pickups += info.get('pickups', 0)
            total_deliveries += info.get('deliveries', 0)
            total_distance += info.get('vehicle_distance', 0)

            obs = next_obs
            steps += 1

        # Episode summary
        all_metrics['total_deliveries'] += len(env_eval.deliveries)
        all_metrics['successful_deliveries'] += sum(d.completed and d.successful for d in env_eval.deliveries)
        all_metrics['failed_deliveries'] += sum(d.completed and not d.successful for d in env_eval.deliveries)
        all_metrics['avg_reward'].append(total_reward.mean())
        all_metrics['avg_pickups'].append(total_pickups / max(1, steps))
        all_metrics['avg_distance'].append(total_distance / max(1, steps))
        all_metrics['vehicle_distances'].append(total_distance)

        for i, ag in enumerate(env_eval.agents):
            all_metrics['distance_per_agent'][i] += ag['distance_travelled']
            all_metrics['deliveries_per_agent'][i] += sum(
                d._just_delivered_by == i for d in env_eval.deliveries if hasattr(d, '_just_delivered_by')
            )

        print(f"Ep {ep}/{episodes} | meanR {total_reward.mean():+.2f} | pickups {total_pickups} | "
              f"deliveries {total_deliveries} | total_dist {total_distance:.1f}")

    # Aggregate summary
    print("\n==== Evaluation Summary ====")
    print(f" Episodes run: {episodes}")
    print(f" Total deliveries: {all_metrics['total_deliveries']}")
    print(f" Successful deliveries: {all_metrics['successful_deliveries']}")
    print(f" Failed deliveries: {all_metrics['failed_deliveries']}")
    print(f" Success rate: {all_metrics['successful_deliveries'] / max(1, all_metrics['total_deliveries']):.3f}")
    print(f" Avg reward/episode: {np.mean(all_metrics['avg_reward']):.3f}")
    print(f" Avg pickups/step: {np.mean(all_metrics['avg_pickups']):.4f}")
    print(f" Avg deliveries/step: {np.mean(all_metrics['avg_distance']):.4f}")
    print(f" Mean total vehicle distance/episode: {np.mean(all_metrics['vehicle_distances']):.1f}")
    print("\nPer-agent stats:")
    for i in range(n_agents):
        print(f"  Agent {i}: distance={all_metrics['distance_per_agent'][i]:.1f}, "
              f"deliveries={all_metrics['deliveries_per_agent'][i]:.0f}")

    return all_metrics

In [53]:
# ===== Run evaluation =====
test_metrics = evaluate_maddpg(
    maddpg,
    test_dl,
    adjacency_matrix,
    node_positions,
    episodes=5,         # run a few test episodes
    max_steps=1800,
    n_agents=NUM_AGENTS,
    capacity=2
)

Ep 1/5 | meanR -16.81 | pickups 8 | deliveries 0 | total_dist 328.3
Ep 2/5 | meanR -16.83 | pickups 8 | deliveries 0 | total_dist 272.3
Ep 3/5 | meanR -16.79 | pickups 8 | deliveries 0 | total_dist 417.2
Ep 4/5 | meanR -16.81 | pickups 8 | deliveries 0 | total_dist 358.8
Ep 5/5 | meanR -16.81 | pickups 8 | deliveries 0 | total_dist 378.7

==== Evaluation Summary ====
 Episodes run: 5
 Total deliveries: 78620
 Successful deliveries: 0
 Failed deliveries: 0
 Success rate: 0.000
 Avg reward/episode: -16.808
 Avg pickups/step: 0.0044
 Avg deliveries/step: 0.1950
 Mean total vehicle distance/episode: 351.1

Per-agent stats:
  Agent 0: distance=0.2, deliveries=0
  Agent 1: distance=0.3, deliveries=0
  Agent 2: distance=0.2, deliveries=0
  Agent 3: distance=0.2, deliveries=0
